In [1]:
# Install required libraries.
!pip install -q pandas numpy scikit-learn joblib matplotlib seaborn

In [2]:
# Import pandas for handling dataset.
import pandas as pd

# Load the Telco Churn dataset.
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Show first 5 rows.
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Show dataset shape.
print("Dataset shape:", df.shape)

# Show column names and data types.
print(df.info())

# Show missing values in each column.
print(df.isnull().sum())

# Show target column distribution.
print(df["Churn"].value_counts())

Dataset shape: (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling

In [4]:
# Drop customerID because it is only an identifier and not useful for prediction.
df = df.drop(columns=["customerID"])

# Convert TotalCharges column to numeric because it may contain text values.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Convert target column Churn from Yes/No to 1/0.
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Show missing values after conversion.
print(df.isnull().sum())

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [5]:
# Separate input features from target column.
X = df.drop(columns=["Churn"])

# Store target column separately.
y = df["Churn"]

# Print feature shape and target shape.
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


In [6]:
# Select numerical columns.
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Select categorical columns.
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Print numerical columns.
print("Numerical columns:", numeric_features)

# Print categorical columns.
print("Categorical columns:", categorical_features)

Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [7]:
# Import train_test_split for splitting data.
from sklearn.model_selection import train_test_split

# Split data into training and testing parts.
X_train, X_test, y_train, y_test = train_test_split(
    X,                  # Input features.
    y,                  # Target labels.
    test_size=0.20,     # 20% data for testing.
    random_state=42,    # Fixed random state for same result.
    stratify=y          # Keep churn ratio same in train and test.
)

# Print training and testing sizes.
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (5634, 19)
Testing size: (1409, 19)


In [8]:
# Import Pipeline for step-by-step ML workflow.
from sklearn.pipeline import Pipeline

# Import ColumnTransformer for applying different preprocessing to different columns.
from sklearn.compose import ColumnTransformer

# Import SimpleImputer for filling missing values.
from sklearn.impute import SimpleImputer

# Import StandardScaler for scaling numerical values.
from sklearn.preprocessing import StandardScaler

# Import OneHotEncoder for converting text columns into numbers.
from sklearn.preprocessing import OneHotEncoder

# Create numerical preprocessing pipeline.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # Fill missing numeric values with median.
        ("scaler", StandardScaler())                    # Scale numeric values.
    ]
)

# Create categorical preprocessing pipeline.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # Fill missing text values.
        ("encoder", OneHotEncoder(handle_unknown="ignore"))    # Convert text categories to numeric.
    ]
)

# Combine numerical and categorical preprocessing.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),        # Apply numeric pipeline.
        ("cat", categorical_transformer, categorical_features) # Apply categorical pipeline.
    ]
)

In [9]:
# Import Logistic Regression model.
from sklearn.linear_model import LogisticRegression

# Import Random Forest model.
from sklearn.ensemble import RandomForestClassifier

# Import GridSearchCV for hyperparameter tuning.
from sklearn.model_selection import GridSearchCV

# Create full pipeline with preprocessing and model placeholder.
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # First preprocess data.
        ("classifier", LogisticRegression())  # Then apply classifier.
    ]
)

# Define parameter grid for Logistic Regression and Random Forest.
param_grid = [
    {
        "classifier": [LogisticRegression(max_iter=1000, random_state=42)],
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__solver": ["liblinear"]
    },
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 10, 20],
        "classifier__min_samples_split": [2, 5]
    }
]

# Create GridSearchCV object.
grid_search = GridSearchCV(
    estimator=pipeline,   # Use full pipeline.
    param_grid=param_grid, # Try different models and parameters.
    cv=5,                 # Use 5-fold cross-validation.
    scoring="f1",         # Select best model using F1-score.
    n_jobs=-1,            # Use all CPU cores.
    verbose=1             # Show training progress.
)

In [10]:
# Train all models and select the best one.
grid_search.fit(X_train, y_train)

# Store best model pipeline.
best_pipeline = grid_search.best_estimator_

# Print best parameters.
print("Best Parameters:")
print(grid_search.best_params_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Parameters:
{'classifier': LogisticRegression(max_iter=1000, random_state=42), 'classifier__C': 10.0, 'classifier__solver': 'liblinear'}


In [11]:
# Import evaluation metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

# Predict on test data.
y_pred = best_pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, y_pred)

# Calculate precision.
precision = precision_score(y_test, y_pred)

# Calculate recall.
recall = recall_score(y_test, y_pred)

# Calculate F1-score.
f1 = f1_score(y_test, y_pred)

# Print results.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Print detailed classification report.
print(classification_report(y_test, y_pred))

# Print confusion matrix.
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8048261178140526
Precision: 0.6551724137931034
Recall: 0.5588235294117647
F1-score: 0.6031746031746031
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409

[[925 110]
 [165 209]]


In [12]:
# Import joblib for saving model pipeline.
import joblib

# Save the complete preprocessing + model pipeline.
joblib.dump(best_pipeline, "telco_churn_pipeline.joblib")

# Print confirmation.
print("Pipeline saved successfully as telco_churn_pipeline.joblib")

Pipeline saved successfully as telco_churn_pipeline.joblib


In [13]:
# Load saved pipeline.
loaded_pipeline = joblib.load("telco_churn_pipeline.joblib")

# Predict first 5 test examples.
sample_predictions = loaded_pipeline.predict(X_test.head())

# Print predictions.
print(sample_predictions)

[0 1 0 0 0]


In [14]:
# Import files module from Google Colab.
from google.colab import files

# Download saved pipeline.
files.download("telco_churn_pipeline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>